## 1. Import the SANSFitter class

In [ ]:
import sys
import os

# Add parent directory to path to import sans_fitter module
sys.path.insert(0, os.path.abspath('..'))

from sans_fitter import SANSFitter
import numpy as np

# Disable OpenCL if causing issues
os.environ["HAVE_OPENCL"] = "0"
os.environ['SAS_OPEN_CL'] = "none"

## 2. Create a fitter instance and load data

In [ ]:
# Initialize the fitter
fitter = SANSFitter()

# Load experimental data
fitter.load_data('../simulated_sans_data.csv')

## 3. Set the model (model-agnostic!)

You can use any model from SasModels:
- 'cylinder'
- 'sphere'
- 'core_shell_sphere'
- 'ellipsoid'
- ... and many more!

In [ ]:
# Load a cylinder model
fitter.set_model('cylinder')

## 4. View available parameters

In [ ]:
# Display all model parameters with their current values
fitter.get_params()

## 5. Configure fitting parameters

Set initial values and ranges for the parameters you want to fit.

In [ ]:
# Set parameter values and ranges
fitter.set_param('radius', value=5, min=1, max=50, vary=True)
fitter.set_param('length', value=20, min=5, max=100, vary=True)
fitter.set_param('scale', value=0.1, min=0, max=1, vary=True)
fitter.set_param('background', value=0.01, min=0, max=1, vary=True)
fitter.set_param('sld', value=2.0, min=0, max=10, vary=False)
fitter.set_param('sld_solvent', value=3.0, min=0, max=10, vary=False)

# View updated parameters
fitter.get_params()

## 6. Perform the fit using BUMPS (default engine)

In [ ]:
# Fit using BUMPS with the Nelder-Mead simplex method
result = fitter.fit(engine='bumps', method='amoeba')

## 7. Visualize the results

In [ ]:
# Plot data vs fitted model with residuals
fitter.plot_results(show_residuals=True, log_scale=True)

## 8. Alternative: Fit using LMFit engine

You can switch to the LMFit engine for comparison.

In [ ]:
# Create a new fitter instance for LMFit comparison
fitter_lm = SANSFitter()
fitter_lm.load_data('../simulated_sans_data.csv')
fitter_lm.set_model('cylinder')

# Set same parameters
fitter_lm.set_param('radius', value=5, min=1, max=50, vary=True)
fitter_lm.set_param('length', value=20, min=5, max=100, vary=True)
fitter_lm.set_param('scale', value=0.1, min=0, max=1, vary=True)
fitter_lm.set_param('background', value=0.01, min=0, max=1, vary=True)
fitter_lm.set_param('sld', value=2.0, vary=False)
fitter_lm.set_param('sld_solvent', value=3.0, vary=False)

# Fit using LMFit
result_lm = fitter_lm.fit(engine='lmfit', method='leastsq')

In [ ]:
# Plot LMFit results
fitter_lm.plot_results(show_residuals=True, log_scale=True)

## 9. Save results to file

In [ ]:
# Save fit results and fitted curve
fitter.save_results('fit_results_bumps.csv')

## 10. Try a different model (sphere)

Demonstrating the model-agnostic nature of the fitter.

In [ ]:
# Create new fitter for sphere model
fitter_sphere = SANSFitter()
fitter_sphere.load_data('../simulated_sans_data.csv')
fitter_sphere.set_model('sphere')

# Check available parameters for sphere model
fitter_sphere.get_params()

In [ ]:
# Configure sphere parameters
fitter_sphere.set_param('radius', value=20, min=5, max=100, vary=True)
fitter_sphere.set_param('scale', value=0.1, min=0, max=1, vary=True)
fitter_sphere.set_param('background', value=0.01, min=0, max=1, vary=True)
fitter_sphere.set_param('sld', value=2.0, vary=False)
fitter_sphere.set_param('sld_solvent', value=3.0, vary=False)

# Fit
result_sphere = fitter_sphere.fit(engine='bumps')

In [ ]:
# Plot sphere fit results
fitter_sphere.plot_results(show_residuals=True, log_scale=True)

## Summary

The `SANSFitter` class provides:

✓ **Model-agnostic design** - Works with any SasModels model

✓ **Multiple fitting engines** - BUMPS (default) and LMFit support

✓ **User-friendly parameter management** - Easy to configure and view parameters

✓ **Flexible data loading** - Supports CSV, XML, HDF5 via sasdata

✓ **Q-range restriction** - Fit only a chosen [qmin, qmax] window; excluded points stay visible in plots

✓ **Comprehensive visualization** - Automatic plotting with residuals

✓ **Result export** - Save fitted curves and parameters to file

✓ **Polydispersity support** - Apply size distributions to model parameters
  - Multiple distribution types: gaussian, lognormal, schulz, rectangle, boltzmann
  - Easy to enable/disable globally
  - Configure independently for each polydisperse parameter

## 11. Working with Polydispersity

Polydispersity accounts for the size distribution of particles in a sample. Real experimental systems rarely have perfectly uniform particle sizes.

Key concepts:
- **pd_width**: Relative width of the size distribution (0.1 = 10%)
- **pd_type**: Distribution shape ('gaussian', 'lognormal', 'schulz', etc.)
- **pd_n**: Number of quadrature points (accuracy vs. speed tradeoff)
- **pd_nsigma**: Number of standard deviations to include

In [ ]:
# Create a new fitter for polydispersity demonstration
fitter_pd = SANSFitter()
fitter_pd.load_data('../simulated_sans_data.csv')
fitter_pd.set_model('sphere')

# Check which parameters support polydispersity
print(f"Supports polydispersity: {fitter_pd.supports_polydispersity()}")
print(f"Polydisperse parameters: {fitter_pd.get_polydisperse_parameters()}")

In [ ]:
# Configure form factor parameters
fitter_pd.set_param('radius', value=50, min=10, max=100, vary=True)
fitter_pd.set_param('sld', value=4.0, vary=False)
fitter_pd.set_param('sld_solvent', value=1.0, vary=False)
fitter_pd.set_param('scale', value=0.01, min=0.001, max=1, vary=True)
fitter_pd.set_param('background', value=0.001, min=0, max=0.1, vary=True)

# Configure 10% Gaussian polydispersity on radius
fitter_pd.set_pd_param(
    'radius',
    pd_width=0.1,       # 10% size polydispersity
    pd_type='gaussian', # Gaussian distribution
    vary=False          # Keep PD width fixed during fit
)

# Enable polydispersity globally
fitter_pd.enable_polydispersity(True)

# Display polydispersity configuration
fitter_pd.get_pd_params()

In [ ]:
# Fit with polydispersity enabled
result_pd = fitter_pd.fit(engine='bumps', method='amoeba')
print(f"\nFitted with 10% Gaussian polydispersity on radius")

In [ ]:
# Plot results with polydispersity
fitter_pd.plot_results(show_residuals=True, log_scale=True)

## 12. Comparing Distribution Types

Different distribution types may better represent your sample's size distribution.

In [ ]:
# Try lognormal distribution - common for particle size distributions
fitter_lognorm = SANSFitter()
fitter_lognorm.load_data('../simulated_sans_data.csv')
fitter_lognorm.set_model('sphere')

fitter_lognorm.set_param('radius', value=50, min=10, max=100, vary=True)
fitter_lognorm.set_param('sld', value=4.0, vary=False)
fitter_lognorm.set_param('sld_solvent', value=1.0, vary=False)
fitter_lognorm.set_param('scale', value=0.01, min=0.001, max=1, vary=True)
fitter_lognorm.set_param('background', value=0.001, min=0, max=0.1, vary=True)

# Use lognormal distribution (prevents negative sizes, naturally asymmetric)
fitter_lognorm.set_pd_param('radius', pd_width=0.1, pd_type='lognormal')
fitter_lognorm.enable_polydispersity(True)

# Fit
result_lognorm = fitter_lognorm.fit(engine='bumps', method='amoeba')
print(f"Chi-squared (lognormal): {result_lognorm['chisq']:.4f}")

## 13. Polydispersity with Cylinder Model (Multiple Parameters)

For models with multiple size parameters, you can apply polydispersity to each independently.

In [ ]:
# Cylinder has multiple polydisperse parameters
fitter_cyl_pd = SANSFitter()
fitter_cyl_pd.load_data('../simulated_sans_data.csv')
fitter_cyl_pd.set_model('cylinder')

# Show all polydisperse parameters for cylinder
print(f"Polydisperse parameters: {fitter_cyl_pd.get_polydisperse_parameters()}")

# Configure form factor
fitter_cyl_pd.set_param('radius', value=20, min=5, max=50, vary=True)
fitter_cyl_pd.set_param('length', value=400, min=100, max=1000, vary=True)
fitter_cyl_pd.set_param('scale', value=0.1, vary=True)
fitter_cyl_pd.set_param('background', value=0.01, vary=True)
fitter_cyl_pd.set_param('sld', value=2.0, vary=False)
fitter_cyl_pd.set_param('sld_solvent', value=3.0, vary=False)

# Apply polydispersity to BOTH radius and length
fitter_cyl_pd.set_pd_param('radius', pd_width=0.1, pd_type='gaussian')
fitter_cyl_pd.set_pd_param('length', pd_width=0.15, pd_type='gaussian')
fitter_cyl_pd.enable_polydispersity(True)

# Display all PD settings
fitter_cyl_pd.get_pd_params()

In [ ]:
# Fit cylinder with polydispersity on both radius and length
result_cyl_pd = fitter_cyl_pd.fit(engine='bumps', method='amoeba')
print(f"\nFitted cylinder with polydispersity on radius (10%) and length (15%)")

In [ ]:
# Plot cylinder fit with polydispersity
fitter_cyl_pd.plot_results(show_residuals=True, log_scale=True)

## 14. Restricting the Q Range

Real datasets often contain points you do not want to fit:
- **Low Q**: beam-stop spillover or parasitic scattering
- **High Q**: points dominated by incoherent background

`set_q_range(qmin, qmax)` restricts the fit to a Q window without editing the data file:
- Both fitting engines only see points inside the window (χ² included)
- Excluded points remain visible in plots, grayed out as "Excluded Data"
- The CSV export covers only the fitted range and records the window in its header
- `get_q_range()` / `reset_q_range()` inspect and restore the range

In [ ]:
# Create a fitter and restrict the Q range before fitting
fitter_qr = SANSFitter()
fitter_qr.load_data('../simulated_sans_data.csv')
fitter_qr.set_model('sphere')

fitter_qr.set_param('radius', value=20, min=5, max=100, vary=True)
fitter_qr.set_param('scale', value=0.1, min=0, max=1, vary=True)
fitter_qr.set_param('background', value=0.01, min=0, max=1, vary=True)
fitter_qr.set_param('sld', value=2.0, vary=False)
fitter_qr.set_param('sld_solvent', value=3.0, vary=False)

# Fit only 0.01 <= Q <= 0.4 Å⁻¹ (trim low-Q spillover and high-Q background)
fitter_qr.set_q_range(qmin=0.01, qmax=0.4)

result_qr = fitter_qr.fit(engine='bumps', method='amoeba')

In [ ]:
# Plot: the fitted curve and residuals span only the fitted window;
# points outside [0.01, 0.4] are shown grayed out as "Excluded Data"
fitter_qr.plot_results(show_residuals=True, log_scale=True)

In [ ]:
# Inspect the active range, then restore the full range and refit for comparison
print(f"Current Q range: {fitter_qr.get_q_range()}")

fitter_qr.reset_q_range()
result_full = fitter_qr.fit(engine='bumps', method='amoeba')

print(f"\nχ² restricted range: {result_qr['chisq']:.4f}")
print(f"χ² full range:       {result_full['chisq']:.4f}")

## 15. Bayesian Analysis (MCMC posterior sampling)

Beyond point estimates, `fit_bayesian()` samples the full posterior distribution
of the varying parameters with BUMPS' DREAM sampler. The printed summary includes
per-parameter credible intervals and convergence diagnostics (R-hat, ESS), and
five posterior displays become available:

- `plot_posterior_pairs()` — corner plot (marginals + pairwise sample clouds)
- `plot_param_distribution(param)` — marginal posterior for one parameter
- `plot_posterior_predictive()` — 95% credible band over the data
- `plot_param_correlations()` — correlation heatmap
- `plot_trace()` — MCMC chain traces

The sample counts below are kept small so the cell runs quickly; increase
`samples`/`burn` for production-quality posteriors.

In [ ]:
# Set up a fitter and sample the posterior with DREAM
fitter_bayes = SANSFitter()
fitter_bayes.load_data('../simulated_sans_data.csv')
fitter_bayes.set_model('cylinder')
fitter_bayes.set_param('radius', value=20, min=1, max=100, vary=True)
fitter_bayes.set_param('length', value=400, min=10, max=1000, vary=True)
fitter_bayes.set_param('sld', value=4.0, vary=False)
fitter_bayes.set_param('sld_solvent', value=1.0, vary=False)
fitter_bayes.set_param('scale', value=1.0, min=0.1, max=10, vary=True)
fitter_bayes.set_param('background', value=0.001, min=0, max=1, vary=True)

result_bayes = fitter_bayes.fit_bayesian(samples=2000, burn=100)

In [ ]:
# Corner plot: marginal densities on the diagonal, sample clouds below
fitter_bayes.plot_posterior_pairs()

In [ ]:
# Marginal posterior distribution for a single parameter
fitter_bayes.plot_param_distribution('radius')

In [ ]:
# Posterior predictive check: 95% credible band (+ sampled curves) over the data
fitter_bayes.plot_posterior_predictive(style='band+draws', n_draws=30)

In [ ]:
# Parameter correlations and MCMC chain traces
fitter_bayes.plot_param_correlations()
fitter_bayes.plot_trace()

In [ ]:
# Programmatic access to the posterior chain
posterior = fitter_bayes.get_posterior()
print('Sampled parameters:', posterior.labels)
print('Chain shape:', posterior.samples.shape)
print(posterior.format_summary())

# Export the raw chain for external analysis
posterior.save_posterior_csv('posterior_chain.csv')